### Install environment

In [1]:
import importlib.util
if importlib.util.find_spec("flashrank") is None:
    !pip install vllm==0.10.0
    !pip install triton==3.2.0
    !pip install flashrank langchain langchain-community langchain_google_genai openai faiss-cpu langchain_huggingface crawl4ai unidecode pymupdf4llm google-genai
    !pip uninstall -y openai
    !pip install openai==1.90.0
else:
    print("All libs aldready installed")

All libs aldready installed


In [2]:
!wget -q -O ngrok.zip https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok.zip
!mv ngrok /usr/local/bin/ngrok
!chmod +x /usr/local/bin/ngrok
!ps aux | grep ngrok

'wget' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.
'mv' is not recognized as an internal or external command,
operable program or batch file.
'chmod' is not recognized as an internal or external command,
operable program or batch file.
'ps' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
import os
# Fixed, do not change
os.environ["GLOO_SOCKET_NAME"] = "eth0"
os.environ["NCCL_SOCKET_NAME"] = "eth0"
os.environ["VLLM_HOST_IP"] = "127.0.0.1" # Internal ip for data communicate between VLLM components
os.environ["VLLM_USE_V1"] = "0" # T4 have compute capacity of 7.5, it need at least 8.0 to use V1

##### Download package from server

In [4]:
DOMAIN = "http://127.0.0.1:8000"
BASE_PATH = ""

In [5]:
import requests
import io
import tarfile
import shutil
def unpack_folder(data: bytes, path: str):
    if os.path.exists(path): # Remove old code
        shutil.rmtree(path)
    with io.BytesIO(data) as tar_buffer:
        with tarfile.open(fileobj=tar_buffer, mode='r:gz') as tar:
            tar.extractall(path=path)
def unpack_file(data: bytes, path: str):
    if os.path.exists(path):
        os.remove(path)
    with open(path, 'wb') as file:
        file.write(data)
def unpack_list(*names: str):
    # if DOMAIN == "http://127.0.0.1:8000": return
    for name in names:
        if "." in name:
            url = f"{DOMAIN}/package/{name}"
        else:
            url = f"{DOMAIN}/package/{name}"
        data = requests.get(url).content
        if "." in name:
            unpack_file(data, name)
        else:
            unpack_folder(data, name)
unpack_list(
    "school_name.json"
)

### Config

Load environment variable (api keys)

In [6]:
from dotenv import load_dotenv
load_dotenv(f"{BASE_PATH}worker.env")

True

In [7]:
# import shutil, os
# if os.path.exists(f"{BASE_PATH}/data_logs"):
#     shutil.rmtree(f"{BASE_PATH}/data_logs")

Setup ngrok

In [ ]:
NGROK_PORT = 8002
if DOMAIN != "http://127.0.0.1:8000":
    import subprocess
    subprocess.run(["ngrok", "config", "add-authtoken", os.getenv("NGROK_TOKEN_1", "")])
    subprocess.run(["ngrok", "http", str(NGROK_PORT)], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

Login hugging face

In [1]:
cmd = [
    "hf", "auth", "login",
    "--token", os.getenv("HUGGING_FACE_TOKEN")
]
import subprocess
subprocess.run(cmd)
print("")

NameError: name 'os' is not defined

##### Config

In [10]:
from data_retriever import *
from server import *
from school_mapper import SchoolMapper
from typing import AsyncGenerator, NotRequired, Protocol
from typing import Callable, AsyncGenerator
from openai import AsyncOpenAI, OpenAI
from google import genai
from google.genai import types
import os
import pickle
import json
import asyncio
import enum
import traceback
import copy

In [11]:
from typing import Protocol, AsyncGenerator, TypedDict
class KeywordInfo(TypedDict):
    query: str
    priority: float
    info: str
    school: str
class KeywordModelProtocol(Protocol):
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]: ...

In [12]:
MODEL_ID = "Qwen/Qwen3-4B"
# Retriever config
search_config = WebsearchConfig(
    page_timeout=15,
    file_timeout=15,
)
rag_config = RagConfig(
    embedding_name="intfloat/multilingual-e5-small",
    device="cuda"
)
splitter_config = SplitterConfig(
    tokenizer_name=MODEL_ID,
    chunk_size=512,
    chunk_overlap=0,
    # device="cuda"
)
table_merge_config = MergeTableConfig(
    k_max_previous=5,
    k_max_next=5
)
neighbor_config = MergeNeighborConfig(
    k_previous_chunks=1,
    k_next_chunks=1
)
# Sampling Params
PAGE_RERANKER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 4096
}
KEYWORDS_PARAMS = {
    "temperature": 0.5,
    "top_p": 0.9,
    "max_tokens": 4096
}
ROUTER_PARAMS = {
    "temperature": 0.7,
    "top_p": 0.9,
    "max_tokens": 1024
}
MODELS: list[ModelInfo] = [
    {
        "name": "GPT 4o mini",
        "id": "gpt-4o-mini"
    },
    {
        "name": "Gemini 2.5 flash",
        "id": "gemini-2.5-flash"
    }
]
CLIENT_INFO: WorkerServerInfo = {
    "name": "Test Reader only",
    "domain": "http://127.0.0.1:8002", # Auto change when run with ngrok
    "models": MODELS
}

##### Template, instruction, prefix

In [13]:
from instruction import *

### Utility class

##### Data Retriever

In [14]:
class Retriever:
    """Search in web"""
    def __init__(self, llm_ranker: PageRerankModelProtocol, llm_keywords: KeywordModelProtocol) -> None:
        self.pipeline = DataRetrieverPipeline(
            llm_ranker,
            websearch_config=search_config,
            rag_config=rag_config,
            splitter_config=splitter_config,
            neighbor_merge_config=neighbor_config,
            table_merge_config=table_merge_config
        )
        self.llm_keywords = llm_keywords
        self.school_mapper = SchoolMapper(f"{BASE_PATH}school_name.json")
    async def start(self):
        """Initialize websearch"""
        await self.pipeline.start()
    async def retrive(self, question: str, params: GenerationParams) -> tuple[list[WebSource], list[RagSource]]:
        data = await self.llm_keywords.keywords(question, params)
        queries = []
        school_restrict = params.get("school_domain", False)
        for item in data:
            if not school_restrict:
                queries.append(item["query"])
            else:
                school = item["school"]
                if school.strip() != "":
                    school_domains = self.school_mapper.domains_from_auto(school, 5)[:10]
                    print(f"[DOMAINS]", school_domains)
                    if len(school_domains) > 0:
                        queries.append([item["query"], school_domains])
        return await self.pipeline.retrieve(params, queries)

##### Client to call model

In [15]:
class APIModel:
    def __init__(self) -> None:
        self.gpt_client = AsyncOpenAI(api_key=os.getenv("OPEN_AI_API_KEY"))
        self.gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
        self.logger = CmdLogger("Model")
    async def call(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        print(f"[API] {call_type} | Instruction length: {len(instruction)} | Prompt length: {len(prompt)} | kwargs: {params.get('kwargs')}")
        model_id = params["model_id"]
        if "gpt" in model_id:
            stream = await self.gpt_client.chat.completions.create(
                model=model_id, 
                messages=[
                    {"role": "system", "content": instruction},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=params.get("max_tokens", 4096),
                temperature=params.get("temperature", 0.7),
                top_p=params.get("top_p", 0.9),
                presence_penalty=params.get("presence_penalty", 0.1),
                frequency_penalty=params.get("frequency_penalty", 0.0),
                stream=True
            )
            total_text = ""
            async for event in stream:
                chunk = event.choices[0].delta.content
                if chunk is not None:
                    total_text += chunk
                    yield chunk
        else:
            gemini_config = types.GenerateContentConfig(
                temperature=params.get("temperature", 0.8),
                top_p=params.get("top_p", 0.9),
                top_k=params.get("top_k", 16),
                max_output_tokens=params.get("max_tokens", 4048)
            )
            stream = await self.gemini_client.aio.models.generate_content_stream(
                model=model_id,
                contents=[
                    types.Content(
                        role="user",
                        parts=[types.Part.from_text(text=instruction)]
                    ),
                    types.Content(
                        role="user",
                        parts=[types.Part.from_text(text=prompt)]
                    )
                ],
                config=gemini_config,
            )
            async for chunk in stream:
                if chunk.candidates:
                    if chunk.candidates[0].content:
                        if chunk.candidates[0].content.parts:
                            for part in chunk.candidates[0].content.parts:
                                if part.text:
                                    yield part.text
    async def __call__(self, call_type: CallType, instruction: str, prompt: str, params: GenerationParams) -> AsyncGenerator[str, None]:
        return self.call(call_type, instruction, prompt, params)
    async def rerank_page(self, pages: list[SearchResult], query: str, relative_threshold: float, params: GenerationParams) -> list[SearchResult]:
        if len(pages) == 0: return []
        text = ""    
        scores = [0.0 for _ in pages]
        prompt = self._construct_reranker_prompt(query, pages)
        copy_params = copy.deepcopy(params)
        copy_params.update(PAGE_RERANKER_PARAMS) #type:ignore
        async for chunk in await self(
            call_type=CallType.RANKER, 
            instruction=PAGE_RERANKER_INSTRUCTION, 
            prompt=PAGE_RERANKER_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            if "```" in text:
                text = text.replace("```", "").strip()
            if text[:4] == "json":
                text = text[4:]
            result = json.loads(text)
            if "output" in result:
                for item in result["output"]:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
            else:
                # Fallback if model not provide intermediate step
                for item in result:
                    index = int(item["index"]) - 1
                    scores[index] = float(item["score"])
        except:
            print(text)
            traceback.print_exc()
        self.logger.log("-----Original-----")
        for page in pages:
            self.logger.log(f"{page["score"]:.3f} + {page["title"]}")
        max_score = 0
        for score, search_result in zip(scores, pages):
            max_score = max(max_score, score)
            search_result["score"] = score
        if max_score == 0: return []
        
        results: list[SearchResult] = []
        for search_result in pages:
            if search_result["score"] >= max_score * relative_threshold:
                results.append(search_result)
        results = sorted(results, key=lambda r:r["score"], reverse=True)
        self.logger.log("-----Reorder-----")
        for page in results:
            self.logger.log(f"{page["score"]:.3f} + {page["title"]}")
        return results
    def _construct_reranker_prompt(self, query: str, data: list[SearchResult]) -> str:
        candidates = [{
            "index": index+1, 
            "title": item["title"],
            "description": item["description"],
        } for index, item in enumerate(data)]
        candidates = "[" + ",\n".join([json.dumps(item, ensure_ascii=False) for item in candidates]) + "]"
        return PAGE_RERANKER_TEMPLATE.format(query=query, pages=candidates)
    async def keywords(self, question: str, params: GenerationParams, threshold: float = 0.5) -> list[KeywordInfo]:
        copy_params = copy.deepcopy(params)
        copy_params.update(KEYWORDS_PARAMS) #type:ignore
        prompt = KEYWORD_TEMPLATE.format(question=question)
        text = ""
        async for chunk in await self(
            call_type=CallType.KEYWORDS, 
            instruction=KEYWORDS_INTRUCTION, 
            prompt=KEYWORDS_PREFIX+prompt, 
            params=copy_params
        ):
            text += chunk
        try:
            if "```" in text:
                text = text.replace("```", "").strip()
            if text[:4] == "json":
                text = text[4:]
            result: list[KeywordInfo] = json.loads(text)
            for item in result:
                self.logger.log(item)
            return result
        except:
            print(text)
            traceback.print_exc()
            return []

##### Pipeline

In [16]:
class CombinedProtocol(ModelProtocol, KeywordModelProtocol, PageRerankModelProtocol):
    pass
class CustomQA:
    def __init__(self, model_protocol: CombinedProtocol) -> None:
        self.logger = CmdLogger("QA")
        self.retriever = Retriever(model_protocol, model_protocol)
        self.llm_call = model_protocol
    async def start(self):
        await self.retriever.start()
    async def inference(self, prompt: str, request: WorkerChatRequest) -> AsyncGenerator[str, None]:
        text = ""
        async for chunk in await self.llm_call(
            call_type=CallType.READER, 
            instruction=READER_UNTRAINED_INSTRUCTION+READER_UNTRAINED_PREFIX, 
            prompt=prompt, 
            params=request["params"]
        ):
            text += chunk
            yield chunk
    async def pre_inference(
        self,
        question: str,
        stream_id: str,
        params: GenerationParams
    ) -> tuple[str, ModelPreOutput]:
        web_sources, rag_sources = await self.retriever.retrive(
            question, 
            params
        )
        context = SourceFormat()(rag_sources)
        prompt = READER_TEMPLATE.format(context=context, question=question)
        self.logger.start()
        pre_output: ModelPreOutput = {
            "generation_params": params,
            "web_sources": web_sources,
            "rag_sources": rag_sources,
            "extra_data": {
            },
            "result_url": stream_id,
        }
        return prompt, pre_output

### Final

VLLMM Engine

In [17]:
api_model = APIModel()

Websearch pipeline

In [18]:
ws_pipeline = CustomQA(api_model)
await ws_pipeline.start()
import uuid
REQUEST_STORAGE: dict[str, tuple[str, WorkerChatRequest, ModelPreOutput]] = {}
async def pre_inference_function(request: WorkerChatRequest) -> ModelPreOutput:
    stream_id = str(uuid.uuid4())
    params = request["params"]
    params["school_domain"] = True #True
    params["engine_type"]= "brave"
    params["page_score_threshold"] = 0.51
    params["merge_table"] = True
    prompt, pre_output = await ws_pipeline.pre_inference(
        request["text"],
        stream_id,
        request["params"]
    ) 
    REQUEST_STORAGE[stream_id] = (prompt, request, pre_output)
    return pre_output

async def inference_function(stream_id: str) -> AsyncGenerator[str, None]:
    prompt, request, pre_output = REQUEST_STORAGE.pop(stream_id)
    generator = ws_pipeline.inference(prompt, request)
    total = ""
    try:
        async for chunk in generator:
            total += chunk
            yield chunk
    finally:
        # Store chat data when finish
        model_output: ModelOutput = {
            **pre_output,
            "text": total
        }
        data: WorkerStoreChatData = {
            "forward_kwargs": request["forward_kwargs"],
            "model_output": model_output
        }
        await app.state.store_chat(data)

Connect to server

In [ ]:
app = construct_app(
    server_domain=DOMAIN,
    info=CLIENT_INFO,
    pre_inference=pre_inference_function,
    inferece=inference_function,
    init_tasks=[],
    shutdown_tasks=[],
    is_local=True
)
# CORS policy
from fastapi.middleware.cors import CORSMiddleware
origins = [
    "http://127.0.0.1:8000",
    DOMAIN
]
app.add_middleware(
    CORSMiddleware,
    allow_origins=origins,
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)
import nest_asyncio
import uvicorn
nest_asyncio.apply()
uvicorn.run(app, port=NGROK_PORT)

INFO:     Started server process [30164]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8002 (Press CTRL+C to quit)


[API] CallType.KEYWORDS | Instruction length: 168 | Prompt length: 2740 | kwargs: None


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


[Model] {'query': 'Điểm chuẩn Trường Đại học Công nghệ - Đại học Quốc gia Hà Nội 2025', 'priority': 1.0, 'info': 'tên trường, năm, điểm chuẩn', 'school': 'đại học công nghệ - đại học quốc gia hà nội'}
[Model] {'query': 'Thông báo tuyển sinh Trường Đại học Công nghệ - Đại học Quốc gia Hà Nội 2025', 'priority': 0.8, 'info': 'tên trường, năm, thông báo tuyển sinh', 'school': 'đại học công nghệ - đại học quốc gia hà nội'}
[DOMAINS] ['uet.vnu.edu.vn']
[DOMAINS] ['uet.vnu.edu.vn']
[Retriever] Websearch: 1.38541s
[API] CallType.RANKER | Instruction length: 100 | Prompt length: 2362 | kwargs: None


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

c:\Users\Anh\.conda\envs\ai\Lib\site-packages\torch\nn\modules\module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.
INFO:faiss:Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
c:\Users\Anh\.conda\envs\ai\Lib\site-packages\torch\nn\modules\module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[Retriever] RAG: 3.28151s
Split 2197 characters to 2 chunks
Split Trường Đại học Công nghệ, ĐHQGHN - VNU- University of Engineering and Technology to 2 chunks


c:\Users\Anh\.conda\envs\ai\Lib\site-packages\torch\nn\modules\module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


[Retriever] RAG: 0.69175s
https://uet.vnu.edu.vn/truong-dai-hoc-cong-nghe-dhqghn-ma-truong-qhi-thong-tin-tuyen-sinh-dai-hoc-nam-2025/
https://tuyensinh.uet.vnu.edu.vn/tin-tuyen-sinh/thong-bao-diem-trung-tuyen-vao-truong-dai-hoc-cong-nghe-dhqghn-nam-2024-theo-phuong-thuc-thpt/
INFO:     127.0.0.1:58891 - "POST /pre_inference HTTP/1.1" 200 OK
INFO:     127.0.0.1:64930 - "POST /inference/93e1111e-659f-45cd-b12b-e20ea22b7021 HTTP/1.1" 200 OK
[API] CallType.READER | Instruction length: 4669 | Prompt length: 10406 | kwargs: None


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


###### Note 
If the instruction is too long -> freeze. Seem like vLLM cache instruction, but we still don't know why it only freeze after some requests. (Seem like cache problem, use llm serve would cause it)
